# Approved Cohort Coverage

Master list of the BMI verified source variables.

Which approved input lists do we need?
|Input list	|Recommendation	|Why|
| --- | --- | ---- |
|approved_variables	|Yes — start here	Records each verified variable, its meaning, and Codebook citation.|
|approved_wave_coverage	|Yes, after variable verification	Records which variables apply to which waves and the supporting evidence.|
|approved_cohort_coverage	|Only if needed	Records cohort availability or applicability. Cohort membership may be better established from respondent data and documented cohort definitions.|
|approved_statistics	|Not as an input approval list	Calculate BMI means and other statistics from the validated survey data. Use Codebook descriptive statistics as a validation reference when appropriate.|
|approved_value_labels	|If needed	Captures documented codes and labels for categorical variables such as race and gender/sex.|
|approved_chunks	|Useful for the AI Agent, but separate	Preserves searchable Codebook passages for retrieval and traceability. It is not a list of approved analytical variables.|

### Two design considerations
- Coverage is not approval.  Existing approval decisions, including Rejected, are preserved by the merge. Recalculation updates the count and computed coverage status, not the human approval fields.
- This computes **non-null records**, not unique respondents. If the Bronze table contains multiple records per respondent, non_null_record_count counts each qualifying record. If your business definition requires unique respondents, we should instead count distinct respondent identifiers.

### Invariant Approved Variables
Invariant variables, like HACOHORT, RARACEM, or RAGENDER will not have wave_numbers.
The "wave_number" will be set to "NONE" because the apply to all waves.


### Data Relationships
1. For each Approved Variable, you will see 1 to 16 Waves
2. Foreach Wave, you will see up to 8 Cohorts + the non_null_record_count.
3. If the non_null_record_count = 0, then data was not captured for that variable and the "cohort_coverage" will set to "NO_DATA".

### Configuration

In [0]:
# ============================================================
# 1.1 Load Libraries
# ============================================================

from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, LongType
)

# ------------------------------------------------------------
# 1.2 Configuration
# ------------------------------------------------------------

source_table = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

approve_table = ("dev_catalog.hrs_agent_metadata.hrs_use_case_approved_variables")

coverage_table = ("dev_catalog.hrs_agent_metadata.hrs_use_case_approved_cohort_coverage")

# Use case is a run parameter — select it via the widget (or the job
# parameter when scheduled) instead of editing this cell for each use case.
dbutils.widgets.text("use_case_name", "bmi_statistics_by_wave_race_gender_age_cohort", "Use case name")
dbutils.widgets.text("codebook_version", "1992-2022v1", "Codebook version")

use_case_name = dbutils.widgets.get("use_case_name")
codebook_version = dbutils.widgets.get("codebook_version")

# Source cohort column
cohort_column = "hacohort"

print(f"Source table: {source_table}")
print(f"Cohort column: {cohort_column}")
print(f"Use case name: {use_case_name}")
print(f"Codebook version: {codebook_version}")

### 2. Read approved variables and Bronze schema

In [0]:
# ------------------------------------------------------------
# 2. Read approved variables and Bronze schema
#
# Display each approved variable and wave number for it.
# ------------------------------------------------------------

approved_df = (
    spark.table(approve_table)
    .filter(F.col("use_case_name") == use_case_name)
    .filter(F.col("codebook_version") == codebook_version)
    .filter(F.col("approval_status") == "Approved")
    .select(
        "variable_name",
        "wave_number"
    )
    .dropDuplicates(["variable_name", "wave_number"])
)

approved_variables = approved_df.collect()

if not approved_variables:
    raise ValueError(
        "No approved variables found for this use case and Codebook version."
    )

# Read Bronze table
source_df = spark.table(source_table)

# Resolve column names case-insensitively.
source_column_lookup = {
    column.lower(): column
    for column in source_df.columns
}

if cohort_column.lower() not in source_column_lookup:
    raise ValueError(
        f"Source table does not contain cohort column '{cohort_column}'."
    )

actual_cohort_column = source_column_lookup[cohort_column.lower()]

# Verify that every approved variable exists in Bronze.
missing_variables = sorted({
    row["variable_name"]
    for row in approved_variables
    if row["variable_name"].lower() not in source_column_lookup
})

if missing_variables:
    raise ValueError(
        "Approved variables missing from Bronze: "
        + ", ".join(missing_variables)
    )

print(f"Approved variable-wave combinations: {len(approved_variables)}")
display(approved_variables)


### 3. Calculate non-null record counts by cohort

Identify each approved variable by Cohort and Wave relationships.  The variable was captured by the Wave survey (HAS DATA) or it was not caputured the Wave survey (NO DATA).
Fore Exmaple
* R1BMI
    * HAS Cohort DATA captured in WAVE #3
    * NO Cohort DATA captured in WAVES #1,2,4,5,6,7, and 8 
* R16BMI
    * HAS Cohort DATA captured in WAVES #1,2,3, 4,5,6,7, and 8 (all)

#### Non-Null Record Counts

This represents the number of respondent rows in the Bronze (source) table that have an actual value for that variable in that wave, broken out by Cohort.


In [0]:
# ------------------------------------------------------------
# 3. Calculate non-null record counts by cohort
#
# Identify each approved variable by Cohort and Wave relationships.
# ------------------------------------------------------------

coverage_frames = []

for row in approved_variables:
    variable_name = row["variable_name"]
    wave_number = row["wave_number"]

    actual_variable_column = source_column_lookup[
        variable_name.lower()
    ]

    # Count source records with a non-null value for this variable.
    variable_coverage = (
        source_df
        .filter(F.col(actual_cohort_column).isNotNull())
        .groupBy(F.col(actual_cohort_column).alias("cohort_number"))
        .agg(
            F.count(
                F.col(actual_variable_column)
            ).cast("long").alias("non_null_record_count")
        )
        ## create iterals values for use case, codebook version, variable name, wave number
        .withColumn("use_case_name", F.lit(use_case_name))
        .withColumn("codebook_version", F.lit(codebook_version))
        .withColumn("variable_name", F.lit(variable_name))
        .withColumn(
            "wave_number",
            F.lit(
                int(wave_number) if wave_number is not None else None
            ).cast("int")
        )
        .withColumn(
            "coverage_status",
            F.when(
                F.col("non_null_record_count") > 0,
                F.lit("HAS_DATA")
            ).otherwise(F.lit("NO_DATA"))
        )
        .withColumn(
            "cohort_number",
            F.col("cohort_number").cast("double")
        )
        .select(
            "use_case_name",
            "codebook_version",
            "variable_name",
            "wave_number",
            "cohort_number",
            "non_null_record_count",
            "coverage_status"
        )
    )

    coverage_frames.append(variable_coverage)

coverage_schema = StructType([
    StructField("use_case_name", StringType(), False),
    StructField("codebook_version", StringType(), False),
    StructField("variable_name", StringType(), False),
    StructField("wave_number", IntegerType(), True),
    StructField("cohort_number", DoubleType(), True),
    StructField("non_null_record_count", LongType(), False),
    StructField("coverage_status", StringType(), False)
])

coverage_df = reduce(
    lambda left, right: left.unionByName(right),
    coverage_frames
)

display(
    coverage_df.orderBy(
        "variable_name",
        "cohort_number"
    )
)



### 4. Validate the calculated coverage records
pre-write validation gate — it runs between computing coverage_df (previous step) and the  (next step), and it enforces two fail-fast checks that protect the coverage table from a bad write:

1. Duplicate-key check. 
* It groups the computed coverage records by the five identity columns — use_case_name, codebook_version, variable_name, wave_number, cohort_number — and if any key appears more than once, it displays the offending rows and raises a ValueError, aborting before anything is written. This matters because the MERGE joins on exactly those five columns: duplicate source keys would make one target row match multiple source rows, causing the MERGE to fail non-deterministically mid-transaction — this way you get a clear failure with the guilty rows shown, instead of a cryptic write error or silently corrupted counts.

2. Target-schema check. 
* It compares the live coverage table's columns against the 11 expected columns (the 7 data columns plus the 4 audit columns: reviewed_by, reviewed_at, created_at, updated_at) and raises a ValueError naming any missing ones. This guards against schema drift — if the table was altered or created by an older version of the DDL, the MERGE would fail with an opaque error; this check catches it first with an actionable message.

#### So its purpose is: 
Only clean, uniquely-keyed, schema-compatible coverage data gets persisted. It's also why the run we did appeared to produce "no output" from that cell — both checks passed, so it raised nothing and displayed nothing. That silent success is the design working as intended.



In [0]:
# ------------------------------------------------------------
# 4. Validate the calculated coverage records
# ------------------------------------------------------------

duplicate_coverage = (
    coverage_df
    .groupBy(
        "use_case_name",
        "codebook_version",
        "variable_name",
        "wave_number",
        "cohort_number"
    )
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_coverage.limit(1).count() > 0:
    display(duplicate_coverage)
    raise ValueError("Duplicate coverage keys found.")

# Ensure the target table contains the expected columns.
target_columns = {
    column.lower()
    for column in spark.table(coverage_table).columns
}

required_columns = {
    "use_case_name",
    "codebook_version",
    "variable_name",
    "wave_number",
    "cohort_number",
    "non_null_record_count",
    "coverage_status",
    "reviewed_by",
    "reviewed_at",
    "created_at",
    "updated_at"
}

missing_columns = required_columns - target_columns

if missing_columns:
    raise ValueError(
        "Coverage table is missing required columns: "
        + ", ".join(sorted(missing_columns))
    )



### 5. Upsert coverage records into Delta

writes the validated coverage records into hrs_use_case_approved_cohort_coverage using an idempotent (repeating action) upsert (MERGE). It does four things worth understanding:

1. It stages the computation as a MERGE source. coverage_df (the in-memory result of cell 7's calculation, validated by cell 9) is registered as a temp view, because Delta's MERGE needs a queryable source object.

2. It matches on the same five-column composite key — use_case_name + codebook_version + variable_name + wave_number + cohort_number — exactly the key cell 9 proved to be unique, so every computed record matches at most one existing row.

3. The one subtle but critical detail: wave_number uses the null-safe operator <=>, not =. Invariant variables (RACEM, HACOHORT, GENDER, BYEAR) have NULL wave_number, and in SQL NULL = NULL is never true — with a plain =, every re-run would fail to match those rows and insert duplicates instead. <=> treats two NULLs as equal, so invariant coverage rows are correctly recognized as existing and updated.

4. The two branches deliberately treat history differently:

* WHEN MATCHED (re-run / re-measurement): refreshes only non_null_record_count, coverage_status, and updated_at. It leaves created_at and — importantly — reviewed_by/reviewed_at untouched, so re-running never overwrites a human reviewer's decisions on existing rows.
* WHEN NOT MATCHED (new combination): inserts the row with both timestamps stamped and the review columns NULL — flagging it as computed but not yet reviewed.

Why this design: it makes the whole notebook safely re-runnable — re-running after a Codebook update or new data load refreshes the evidence without duplicating rows or destroying audit history. You saw exactly that behavior in the run we did: the 180 pre-existing rows kept their 2026-10-22 created_at while their updated_at moved to today, and the 288 new HEIGHT/WEIGHT rows were inserted fresh with NULL review fields.

In [0]:
# ------------------------------------------------------------
# 5. Upsert coverage records into Delta
# ------------------------------------------------------------

coverage_df.createOrReplaceTempView("coverage_updates")

spark.sql(f"""
    MERGE INTO {coverage_table} AS target
    USING coverage_updates AS source
      ON target.use_case_name = source.use_case_name
     AND target.codebook_version = source.codebook_version
     AND target.variable_name = source.variable_name
     AND target.wave_number <=> source.wave_number
     AND target.cohort_number = source.cohort_number

    WHEN MATCHED THEN UPDATE SET
        target.non_null_record_count = source.non_null_record_count,
        target.coverage_status = source.coverage_status,
        target.updated_at = current_timestamp()

    WHEN NOT MATCHED THEN INSERT (
        use_case_name,
        codebook_version,
        variable_name,
        wave_number,
        cohort_number,
        non_null_record_count,
        coverage_status,
        reviewed_by,
        reviewed_at,
        created_at,
        updated_at
    )
    VALUES (
        source.use_case_name,
        source.codebook_version,
        source.variable_name,
        source.wave_number,
        source.cohort_number,
        source.non_null_record_count,
        source.coverage_status,
        NULL,
        NULL,
        current_timestamp(),
        current_timestamp()
    )
""")

print("Cohort coverage records upserted successfully.")




### Verify persisted coverage

post-write verification step — the read-back checkpoint that confirms what actually landed in the coverage table. It does this in two displays:

1. The full persisted listing. 
* It reads from the Delta table itself — spark.table(coverage_table) filtered to this use case and codebook version — not from the in-memory coverage_df. 
* That's the key design decision: it verifies the source of truth after the write, rather than re-displaying what the computation intended to write. So it would catch write-path failures that the in-memory validations in cells 9–11 can't: rows the MERGE failed to insert, rows matched to the wrong targets, or statuses that didn't persist — and because the display includes the audit columns, you can see at a glance how each row arrived (in our run, this is how we confirmed the idempotent MERGE behavior: existing rows showing created_at 2026-10-22 with a fresh updated_at, and the new HEIGHT/WEIGHT rows stamped today).

2. The one-glance summary rollup. A GROUP BY coverage_status showing record count and summed non_null_record_count per status — the quick sanity check that the totals look right before sign-off. In the run we did, that displayed: HAS_DATA → 321 records / 1,060,043 non-null records; NO_DATA → 147 records / 0 — which my independent SQL check reproduced exactly.

Purpose in the pipeline: it's the human checkpoint that closes the notebook — the analyst looks at these two outputs (row-level detail if anything looks off, summary for the go/no-go) before treating the coverage evidence as authoritative for downstream use. It's deliberately read-only and safe to re-run at any time.

Together, the closing trio works as: cell 9 validates before writing, cell 11 writes, cell 13 proves the write — a validate → persist → verify discipline where each step's failure mode is caught by the next.

In [0]:
# ------------------------------------------------------------
# 7. Verify persisted coverage
# ------------------------------------------------------------

persisted_coverage = (
    spark.table(coverage_table)
    .filter(F.col("use_case_name") == use_case_name)
    .filter(F.col("codebook_version") == codebook_version)
)

display(
    persisted_coverage
    .orderBy(
        "variable_name",
        "cohort_number"
    )
)


# Summary for quick validation
display(
    persisted_coverage
    .groupBy("coverage_status")
    .agg(
        F.count("*").alias("coverage_record_count"),
        F.sum("non_null_record_count").alias(
            "total_non_null_records"
        )
    )
    .orderBy("coverage_status")
)


In [0]:
# OPTIONAL - D. approved_statistics — keep separate from approvals (Not as an input approval list)

bmi_validation_statistics = [
    {
        "variable_name": "R1BMI",
        "statistic_name": "Mean",
        "reference_value": None,       # Populate from verified source
        "source_page": 575
    }
]

In [0]:
# OPTIONAL - E. approved_value_labels — use for categorical variables (If needed)
approved_value_labels = [
    {
        "variable_name": None,         # Verified source column
        "value_code": None,
        "value_label": None,
        "wave_number": None,
        "source_page": None
    }
]